# Musanga V2 — 01 | Reference data and spatial partitions

**Purpose:** Establish the frozen labels and independent geographic validation protocol before testing illumination-robust U-Nets. Run after [V2 Notebook 00](00_UAV_Input_Data_Audit.ipynb), which must first produce `00_Luki_merged_evaluation_crowns.gpkg`.

**Data:** Four original training forests (MOS_1_2, Yoko, Yangambi, Baego) at **native UAV resolution** with the original `Class_ID == 1` Musanga interpretation and crowns **≥5 m²**. Luki is **external validation only**, containing the verified **364** crowns: **179 Musanga** and **185 non-Musanga** from Nadine plus the independent field inventory. No Luki crown enters training or model selection.

**Method:** Quarantine invalid crown geometries (do not edit source files); define deterministic 250 m spatial validation blocks with a 30 m polygon-to-polygon separation buffer; retain site-level LOSO tests; measure median luminance within individual crown polygons using only the native RGB bands and define illumination strata from the four training forests, not Luki.

**Output contract:** exclusively `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/01_reference_splits/`. Data, existing UAV mosaics, georeferencing, original labels and legacy outputs are never changed.

**Execution:** CPU-only. Crown RGB sampling in Section 05 can take some time because it reads thousands of small windows from Google Drive.


## 00 — Configuration and immutable dataset contracts

Resolve exact UAV rasters and crown labels, and enforce exclusive V2 output paths.

In [ ]:
%pip -q install geopandas pyogrio rasterio shapely scikit-learn matplotlib pandas
from pathlib import Path
import json,hashlib,math,random
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from shapely.geometry import box
from scipy.spatial import cKDTree
from IPython.display import display
from google.colab import drive
from rasterio.windows import from_bounds,Window
from rasterio.features import geometry_mask
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
AUDIT=V2/"00_input_audit";OUT=V2/"01_reference_splits"
OUT.mkdir(parents=True,exist_ok=True)
assert AUDIT.exists(),f"Run V2 Notebook 00 first: {AUDIT}"
LUKI_SOURCE=AUDIT/"00_Luki_merged_evaluation_crowns.gpkg"
assert LUKI_SOURCE.exists(),f"Required MERGED Luki reference missing; run Notebook 00 Section 03: {LUKI_SOURCE}"
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
LUKI_RGB=GEO/"Luki.tif"
MIN_AREA=5.0;GROUP_M=250.0;BUFFER_M=30.0;VAL_FRACTION=.20;SEED=20261013
# Only numeric annotation Class_ID=1 is Musanga. Other numeric values = non-Musanga.
# Luki is external evaluation ONLY, across all LOSO folds and model selection.
print("Training forests:",list(SITES))
print("Independent Luki evaluation:",LUKI_SOURCE)
print("V2 output:",OUT)


## 01 — Load training crowns and audit geometry

Apply the historical ≥5 m² crown threshold, inspect native geometries and class labels, and quarantine invalid polygons rather than modifying input shapefiles.

In [ ]:
TABLES={};RASTERS={};SUMMARY=[]
for site,(rgb,shp) in SITES.items():
    rp=GEO/rgb;ap=AD/shp
    assert rp.is_file() and ap.is_file(),(site,rp,ap)
    with rasterio.open(rp) as ds:
        assert ds.crs is not None and ds.crs.is_projected
        RASTERS[site]=dict(path=str(rp),crs=str(ds.crs),transform=ds.transform,
                            bounds=ds.bounds,gsd_x=abs(ds.transform.a),gsd_y=abs(ds.transform.e))
    raw=gpd.read_file(ap)
    assert raw.crs is not None,f"{site}: missing annotation CRS"
    assert "Class_ID" in raw,f"{site}: missing Class_ID"
    original=len(raw)
    if str(raw.crs)!=RASTERS[site]["crs"]:raw=raw.to_crs(RASTERS[site]["crs"])
    valid_geom=raw.geometry.notna()&(~raw.geometry.is_empty)&raw.geometry.is_valid
    bad=int((~valid_geom).sum())
    raw=raw.loc[valid_geom].copy()
    numeric=pd.to_numeric(raw.Class_ID,errors="coerce")
    bad_class=int(numeric.isna().sum())
    raw=raw.loc[numeric.notna()].copy()
    raw["class_id"]=numeric.loc[raw.index].astype(int)
    raw["label"]=(raw.class_id==1).astype(np.uint8)
    raw["area_m2"]=raw.geometry.area.astype(float)
    raw=raw.loc[raw.area_m2>=MIN_AREA].copy()
    b=RASTERS[site]["bounds"]
    footprint=box(b.left,b.bottom,b.right,b.top)
    raw=raw.loc[raw.geometry.intersects(footprint)].copy()
    raw["source_row"]=raw.index.astype(str)
    raw=raw.reset_index(drop=True)
    raw["crown_id"]=[f"{site}:{r}" for r in raw.source_row]
    assert raw.crown_id.is_unique
    TABLES[site]=raw
    SUMMARY.append(dict(site=site,original_rows=original,invalid_or_empty_geometry_excluded=bad,
                        invalid_class_excluded=bad_class,retained=len(raw),
                        musanga=int(raw.label.sum()),non_musanga=int((1-raw.label).sum()),
                        native_gsd_m=RASTERS[site]["gsd_x"],
                        annotation_path=str(ap)))
TRAIN_SUM=pd.DataFrame(SUMMARY)
TRAIN_SUM.to_csv(OUT/"01_training_crown_audit.csv",index=False)
display(TRAIN_SUM)
assert all(len(g)>50 and g.label.nunique()==2 for g in TABLES.values())


## 02 — Load the merged Luki reference as an external holdout

Confirm exactly which Nadine and field-inventory crowns enter the Luki evaluation; never incorporate them into a training or tuning fold.

In [ ]:
assert LUKI_RGB.exists()
with rasterio.open(LUKI_RGB) as ds:
    luki_crs=ds.crs;luki_bounds=ds.bounds
    assert str(luki_crs)=="EPSG:32733",f"Unexpected existing Luki CRS {luki_crs}"
LUKI=gpd.read_file(LUKI_SOURCE,layer="evaluation_crowns")
assert LUKI.crs is not None and str(LUKI.crs)=="EPSG:32733"
assert {"source_dataset","source_id","class_id","geometry"}.issubset(LUKI)
assert LUKI.geometry.notna().all() and (~LUKI.geometry.is_empty).all()
assert LUKI.geometry.is_valid.all(),"Check Notebook 00 merged source invalid polygons"
assert LUKI.source_dataset.isin(["nadine","field_inventory"]).all()
assert set(pd.to_numeric(LUKI.class_id).astype(int)).issubset(set([1,2,3,6,7]))
LUKI["label"]=(LUKI.class_id.astype(int)==1).astype(np.uint8)
LUKI["crown_id"]=LUKI.source_dataset.astype(str)+":"+LUKI.source_id.astype(str)
assert LUKI.crown_id.is_unique,"Duplicate provenance IDs in merged Luki dataset"
LUKI["area_m2"]=LUKI.geometry.area
assert (LUKI.area_m2>=MIN_AREA-1e-5).all()
print("Merged Luki holdout:",len(LUKI),"Musanga:",int(LUKI.label.sum()),
      "non-Musanga:",int((1-LUKI.label).sum()))
display(pd.crosstab(LUKI.source_dataset,LUKI.label).rename(columns={0:"non_Musanga",1:"Musanga"}))
assert len(LUKI)==364 and int(LUKI.label.sum())==179 and int((1-LUKI.label).sum())==185,(
    "Luki evaluation census changed from vetted Notebook 00 result; investigate and version deliberately")
LUKI[["source_dataset","source_id","crown_id","class_id","label","area_m2","geometry"]].to_file(
    OUT/"01_Luki_frozen_external_reference.gpkg",layer="external_evaluation",driver="GPKG")


## 03 — Spatially separated internal validation splits

Assign deterministic 250 m blocks, approximately 20% to validation. Remove training crowns lying within 30 m of validation polygons to prevent boundary leakage. Preserve unassigned (buffered) crowns for audit.

In [ ]:
SPLITS={};SPLIT_REPORT=[]
def assign_spatial_split(site,g):
    # Spatial-block split fixed per forest; do not use labels to pick a lucky split.
    c=g.geometry.centroid
    x0=RASTERS[site]["bounds"].left;y0=RASTERS[site]["bounds"].bottom
    groups=(np.floor((c.x-x0)/GROUP_M).astype(int).astype(str)+"_"+
            np.floor((c.y-y0)/GROUP_M).astype(int).astype(str)).to_numpy()
    unique=np.unique(groups)
    rng=np.random.default_rng(SEED+list(SITES).index(site))
    shuffled=unique.copy();rng.shuffle(shuffled)
    n_val=max(1,int(round(len(unique)*VAL_FRACTION)))
    val_groups=set(shuffled[:n_val])
    split=np.where(np.isin(groups,list(val_groups)),"validation","training")
    # Ensure nearby crowns in adjacent grid cells do not leak into training.
    # Build STRtree of validation geometries; distances are actual polygon geometry distances.
    from shapely.strtree import STRtree
    val_geoms=list(g.geometry.iloc[np.flatnonzero(split=="validation")])
    val_tree=STRtree(val_geoms)
    quarantined=0
    for i in np.flatnonzero(split=="training"):
        geo=g.geometry.iloc[i]
        candidate=val_tree.query(geo.buffer(BUFFER_M))
        if any(geo.distance(val_geoms[int(j)])<BUFFER_M for j in candidate):
            split[i]="buffered"
            quarantined+=1
    out=g.copy();out["spatial_block"]=groups;out["split"]=split
    for key in ("training","validation"):
        z=out[out.split==key]
        if not len(z) or z.label.nunique()!=2:
            raise RuntimeError(f"{site}: {key} missing class; adjust SEED or GROUP_M prior to modelling.")
    return out,dict(site=site,blocks=len(unique),validation_blocks=len(val_groups),
                    training=int((split=="training").sum()),validation=int((split=="validation").sum()),
                    buffered=int((split=="buffered").sum()),
                    validation_musanga=int(out.loc[out.split=="validation","label"].sum()),
                    training_musanga=int(out.loc[out.split=="training","label"].sum()))
for site,g in TABLES.items():
    result,diag=assign_spatial_split(site,g);SPLITS[site]=result
    result[["crown_id","source_row","class_id","label","area_m2","spatial_block","split"]].to_csv(
        OUT/f"01_{site}_split_register.csv",index=False)
    SPLIT_REPORT.append(diag)
SPLIT_DF=pd.DataFrame(SPLIT_REPORT)
SPLIT_DF.to_csv(OUT/"01_spatial_split_summary.csv",index=False)
display(SPLIT_DF)


## 04 — Define complete-forest LOSO and Luki external tests

Produce machine-readable manifests for all four independent leave-one-forest-out evaluations and the entirely separate Luki evaluation.

In [ ]:
LOSO=[]
for heldout in SITES:
    train_sites=[x for x in SITES if x!=heldout]
    train_n=int(sum((SPLITS[s].split=="training").sum() for s in train_sites))
    val_n=int(sum((SPLITS[s].split=="validation").sum() for s in train_sites))
    test_n=len(TABLES[heldout])
    LOSO.append(dict(heldout=heldout,training_sites="|".join(train_sites),
                     n_training_crowns=train_n,n_validation_crowns=val_n,
                     n_external_test_crowns=test_n,
                     test_uses_all_eligible_crowns=True,
                     luki_in_training=False))
LOSO_DF=pd.DataFrame(LOSO)
LOSO_DF.to_csv(OUT/"01_LOSO_fold_registry.csv",index=False)
display(LOSO_DF)
print("Luki external evaluation remains untouched for all model development:")
print(len(LUKI),"crowns and",int(LUKI.label.sum()),"Musanga labels.")


## 05 — Crown illumination covariates for stratified evaluation

Measure native UAV luminance within each crown (bounded raster windows), not frozen U-Net probabilities. Treat brightness as an evaluation attribute, not as a trusted species label.

In [ ]:
# Sample raster luminance inside each crown at source-native pixel scale.
# Avoid 4th alpha mask; Luki's alpha/nodata metadata is unreliable.
MAX_CROWN_READ_PX=2048
def crown_light(site,geometry,src):
    # Bound overly large windows; report them for manual handling instead of allocating excessive RAM.
    win=from_bounds(*geometry.bounds,transform=src.transform)
    win=win.round_offsets().round_lengths()
    left=max(0,int(win.col_off));top=max(0,int(win.row_off))
    right=min(src.width,int(math.ceil(win.col_off+win.width)))
    bottom=min(src.height,int(math.ceil(win.row_off+win.height)))
    w=right-left;h=bottom-top
    if w<=0 or h<=0:return np.nan,0,"outside"
    if w>MAX_CROWN_READ_PX or h>MAX_CROWN_READ_PX:return np.nan,0,"large_window"
    window=Window(left,top,w,h)
    rgb=src.read([1,2,3],window=window)
    inside=geometry_mask([geometry],out_shape=(h,w),transform=src.window_transform(window),invert=True)
    valid=(~np.all(rgb==0,axis=0))&(~np.all(rgb==255,axis=0))&(
        ~((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)))&inside
    if valid.sum()<5:return np.nan,int(valid.sum()),"low_support"
    lum=(.2126*rgb[0].astype(np.float32)+.7152*rgb[1].astype(np.float32)+
         .0722*rgb[2].astype(np.float32))/255.
    return float(np.median(lum[valid])),int(valid.sum()),"ok"
LIGHT_REPORT=[]
for site,g in list(SPLITS.items())+[("Luki",LUKI)]:
    raster=GEO/(SITES[site][0] if site!="Luki" else "Luki.tif")
    with rasterio.open(raster) as src:
        for _,rec in g.iterrows():
            brightness,n,status=crown_light(site,rec.geometry,src)
            LIGHT_REPORT.append(dict(site=site,crown_id=rec.crown_id,
                source_dataset=rec.get("source_dataset","legacy_training"),
                split=rec.get("split","independent_external"),
                label=int(rec.label),median_luminance=brightness,
                valid_rgb_pixels=n,qa_status=status))
    print("Illumination crown audit:",site,"completed",len(g),flush=True)
LIGHT_DF=pd.DataFrame(LIGHT_REPORT)
LIGHT_DF.to_csv(OUT/"01_crown_illumination.csv",index=False)
print("Crown brightness coverage by site:")
display(LIGHT_DF.groupby("site").agg(
    total=("crown_id","count"),measured=("median_luminance","count"),
    median_brightness=("median_luminance","median"),
    p10=("median_luminance",lambda x:x.quantile(.1)),
    p90=("median_luminance",lambda x:x.quantile(.9))).round(3))
# Training-derived brightness strata avoid tuning against Luki evaluation labels.
TRAIN_BRIGHT=LIGHT_DF[(LIGHT_DF.site!="Luki")&
                      (LIGHT_DF.split=="training")].median_luminance.dropna()
assert len(TRAIN_BRIGHT)>100
DARK_CUT=float(TRAIN_BRIGHT.quantile(.20))
BRIGHT_CUT=float(TRAIN_BRIGHT.quantile(.80))
LIGHT_DF["illumination_stratum"]=np.where(
    LIGHT_DF.median_luminance.isna(),"unknown",
    np.where(LIGHT_DF.median_luminance<DARK_CUT,"dark",
    np.where(LIGHT_DF.median_luminance>BRIGHT_CUT,"bright","intermediate")))
LIGHT_DF.to_csv(OUT/"01_crown_illumination.csv",index=False)
print("Fixed TRAIN-only illumination cutoffs:",{"dark_below":DARK_CUT,"bright_above":BRIGHT_CUT})
display(pd.crosstab(LIGHT_DF.site,LIGHT_DF.illumination_stratum))


## 06 — Reproducibility manifest and final safeguards

Save all frozen input/split decisions under `Musanga/KVO_work/V2/01_reference_splits/`, with explicit independence of Luki and no alteration of legacy files.

In [ ]:
POLICY=dict(
 stage="V2/01 Reference Data and Spatial Splits",input_audit=str(AUDIT),
 luki_external_crowns=str(LUKI_SOURCE),
 luki_musanga=179,luki_non_musanga=185,
 train_sites=list(SITES),positive_class="Class_ID==1",
 minimum_area_m2=MIN_AREA,block_size_m=GROUP_M,buffer_m=BUFFER_M,
 validation_block_fraction=VAL_FRACTION,seed=SEED,
 train_validation_strata="fixed spatial block assignments plus crown-geometry separation",
 leave_one_site_out="entire site withheld; heldout site's crowns never used for model selection",
 luki_policy="always heldout; Nadine + field Musanga merged evaluation only; zero training exposure",
 threshold_dark_train_p20=DARK_CUT,threshold_bright_train_p80=BRIGHT_CUT,
 frozen_version="V2/01",
 outputs=str(OUT))
(OUT/"01_reference_split_contract.json").write_text(json.dumps(POLICY,indent=2))
print("NOTEBOOK 01 COMPLETE. All output files:",OUT)
print("No UAV image was resampled, relocated, edited or georeferenced.")
print("The original and Nadine annotations were not altered.")
print("Next: V2 Notebook 02 — frozen baseline illumination-stratified evaluation.")
